# 02. Train an agent from scratch

Trains a small PPO agent on `basesWorkers16x16A` for **1 000 000 steps**
(~5 min on a 2024-era CPU), then walks through every artefact the
training loop produced.

GridNet architecture (0.84 M params) is the smallest in the registry,
picked here to keep the smoke run fast. For the thesis-grade single-map
agent (UECD-SingleMap-Best, 4.7 M params, 350 M steps), use
[`experiments/single-map/train_UECD-SingleMap-Best_phase1.slurm`](../experiments/single-map/train_UECD-SingleMap-Best_phase1.slurm)
on a GPU cluster instead.

Prereq: [`00_navigate.ipynb`](00_navigate.ipynb) must be green.

## 1. Train

The CLI is `microrts-agent train`. Every flag has a sane default; we
override only what matters for the smoke run.

| Flag | Value | Why |
|---|---|---|
| `--exp-name` | `notebook-train_s1` | Output dir: `outputs/runs/notebook-train_s1/` |
| `--architecture` | `gridnet` | Smallest in the registry, fastest on CPU |
| `--map` | `basesWorkers16x16A.xml` | Canonical 16x16 thesis map |
| `--total-timesteps` | `1_000_000` | ~5 min on CPU; far below thesis budgets |
| `--num-bot-envs` | `8` | 8 parallel envs vs `RandomBiasedAI` |
| `--num-selfplay-envs` | `0` | No self-play for the smoke |
| `--with-eval` | `False` | Skip in-training eval to save time |
| `--seed` | `1` | Reproducibility |

Full flag list: `microrts-agent train --help`.

In [ ]:
import subprocess

from microrts_agent.paths import PROJECT_ROOT

run_dir = PROJECT_ROOT / "outputs" / "runs" / "notebook-train_s1"
if run_dir.exists():
    print(f"Note: {run_dir} already exists; delete it to re-train from scratch.\n")

result = subprocess.run(
    [
        "microrts-agent",
        "train",
        "--exp-name",
        "notebook-train_s1",
        "--architecture",
        "gridnet",
        "--map",
        "maps/open_competition/basesWorkers16x16A.xml",
        "--total-timesteps",
        "1000000",
        "--num-bot-envs",
        "8",
        "--num-selfplay-envs",
        "0",
        "--num-steps",
        "256",
        "--with-eval",
        "False",
        "--seed",
        "1",
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=900,
)
print(result.stdout[-1500:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-1500:])

## 2. Inspect what was produced

`microrts-agent train` writes everything under `outputs/runs/<exp-name>/`.
Standard layout:

- `config.json`: every hyperparameter, including the defaults you didn't override.
- `agent.pt`: model state dict (inference-ready, same format as shipped agents).
- `checkpoint.pt`: state dict + optimiser + global_step (resume from here).
- `train.log`: per-step textual log (return, length, win rate per opponent).
- `events.out.tfevents.*`: TensorBoard events (`tensorboard --logdir <run>`).

In [ ]:
for path in sorted(run_dir.iterdir()):
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f"  {size_kb:>10.1f} KB  {path.name}")
    else:
        print(f"  {'<dir>':>13}  {path.name}/")

### Inspect `config.json`

Records the full training setup, useful to remember what produced an
agent (or to feed `load_agent_from_config` later for inference).

In [ ]:
import json

with open(run_dir / "config.json") as f:
    cfg = json.load(f)
for k in [
    "exp_name",
    "architecture",
    "total_timesteps",
    "num_bot_envs",
    "num_selfplay_envs",
    "learning_rate",
    "seed",
]:
    print(f"  {k:25s} = {cfg.get(k)}")
print(f"\n(Full config has {len(cfg)} keys.)")

### Inspect `train.log`

Per-rollout line with `step= ret= len= WR[bot=xx%]` tokens. Useful for
back-of-envelope checks; `06_analysis.ipynb` parses it into a DataFrame
and plots the WR curve.

In [ ]:
log_path = run_dir / "train.log"
lines = log_path.read_text().splitlines()
print(f"train.log: {len(lines)} lines\n")
print("-- first 3 step lines --")
for line in [ln for ln in lines if ln.startswith("step=")][:3]:
    print(" ", line[:200])
print("\n-- last 3 step lines --")
for line in [ln for ln in lines if ln.startswith("step=")][-3:]:
    print(" ", line[:200])

## 3. Play 1 game with the just-trained agent

Same CLI as [`01_evaluate.ipynb`](01_evaluate.ipynb), but pointed at our
new run dir. 1 M steps on GridNet is way below thesis budgets, so don't
expect a UECD-grade win rate. The point here is just to confirm the
training -> eval round-trip works end to end.

In [ ]:
result = subprocess.run(
    [
        "microrts-agent",
        "evaluate",
        "--agent",
        str(run_dir),
        "--opponent",
        "RandomBiasedAI",
        "--maps",
        "maps/open_competition/basesWorkers16x16A.xml",
        "--nb_games",
        "2",
        "--max-steps",
        "2000",
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=300,
)
print(result.stdout[-1500:])

## Next steps

- [`06_analysis.ipynb`](06_analysis.ipynb): parse `train.log` and plot the
  win-rate curve of the run we just produced.
- Bigger budgets: bump `--total-timesteps` to 100 M and switch
  `--architecture` to `unet_entity_cbam_deep` to approach UECD scale.
  Needs a GPU; use one of the SLURM scripts under
  [`experiments/single-map/`](../experiments/single-map/) as a template.
- Resume / fine-tune: `--load-model <agent.pt>` warm-starts from
  an existing checkpoint; `--resume <run-dir>` continues a paused run.